# NYC Taxi Data - Loading Pipeline
This notebook downloads, cleans, and loads NYC TLC taxi trip data into a MySQL star schema database for analytics.

**Data Source:** NYC Taxi and Limousine Commission (TLC) Trip Record Data  
**Coverage:** Yellow and Green taxi trips, January 2025 to February 2026  
**Database:** MySQL star schema (nyc_taxi)

> **Note:** This notebook was developed collaboratively with Claude (Anthropic) 
> as a learning exercise. Each code cell is accompanied by plain-language 
> explanations to make the pipeline accessible to students learning data 
> engineering concepts.
## Step 1 - Download Raw Data Files
Download Yellow and Green taxi Parquet files from the NYC TLC website for the period January 2025 to February 2026.

In [7]:
import os
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta

# ── Configuration ──────────────────────────────────────────────
# All paths are relative to where this notebook lives
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("load_data.ipynb"))
RAW_DATA_PATH = os.path.join(NOTEBOOK_DIR, "Raw Dataset")

# TLC base URL for trip data
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

# Start month for download
START_DATE = datetime(2025, 1, 1)

# End date: two months before today (TLC publishes with ~2 month delay)
END_DATE = datetime.today().replace(day=1) - relativedelta(months=2)

# Taxi types to download
TAXI_TYPES = ["yellow", "green"]

# ── Generate month list dynamically ────────────────────────────
def generate_months(start, end):
    """Generate list of YYYY-MM strings from start to end month inclusive."""
    months = []
    current = start
    while current <= end:
        months.append(current.strftime("%Y-%m"))
        current += relativedelta(months=1)
    return months

months = generate_months(START_DATE, END_DATE)
print(f"Months to download: {months[0]} to {months[-1]} ({len(months)} months)")
print(f"Total files to download: {len(months) * len(TAXI_TYPES)}")

# ── Download Function ───────────────────────────────────────────
def download_file(url, save_path):
    """
    Download a file from a URL and save it locally.
    Skips the file if it already exists.
    Prints status for each file.
    """
    if os.path.exists(save_path):
        print(f"  Already exists, skipping: {os.path.basename(save_path)}")
        return

    response = requests.get(url, stream=True)

    if response.status_code == 200:
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  Downloaded: {os.path.basename(save_path)}")
    else:
        print(f"  NOT FOUND (status {response.status_code}): {os.path.basename(save_path)}")

# ── Main Download Loop ──────────────────────────────────────────
os.makedirs(RAW_DATA_PATH, exist_ok=True)

for taxi_type in TAXI_TYPES:
    print(f"\n── {taxi_type.upper()} TAXI ──")
    for month in months:
        filename  = f"{taxi_type}_tripdata_{month}.parquet"
        url       = f"{BASE_URL}/{filename}"
        save_path = os.path.join(RAW_DATA_PATH, filename)
        download_file(url, save_path)

print("\n✓ Download complete.")

Months to download: 2025-01 to 2026-02 (14 months)
Total files to download: 28

── YELLOW TAXI ──
  Downloaded: yellow_tripdata_2025-01.parquet
  Downloaded: yellow_tripdata_2025-02.parquet
  Downloaded: yellow_tripdata_2025-03.parquet
  Downloaded: yellow_tripdata_2025-04.parquet
  Downloaded: yellow_tripdata_2025-05.parquet
  Downloaded: yellow_tripdata_2025-06.parquet
  Downloaded: yellow_tripdata_2025-07.parquet
  Downloaded: yellow_tripdata_2025-08.parquet
  Downloaded: yellow_tripdata_2025-09.parquet
  Downloaded: yellow_tripdata_2025-10.parquet
  Downloaded: yellow_tripdata_2025-11.parquet
  Downloaded: yellow_tripdata_2025-12.parquet
  Downloaded: yellow_tripdata_2026-01.parquet
  Downloaded: yellow_tripdata_2026-02.parquet

── GREEN TAXI ──
  Downloaded: green_tripdata_2025-01.parquet
  Downloaded: green_tripdata_2025-02.parquet
  Downloaded: green_tripdata_2025-03.parquet
  Downloaded: green_tripdata_2025-04.parquet
  Downloaded: green_tripdata_2025-05.parquet
  Downloaded: g

## Step 2 - Explore the Raw Data
Before loading anything into the database, we inspect the raw Parquet files
to understand the column names, data types, and any obvious quality issues.
We check one Yellow file and one Green file since their schemas differ slightly.

In [4]:
import pandas as pd

# Read just the first 1000 rows of each file for a quick look
# No need to load millions of rows just to explore the structure

yellow_sample = pd.read_parquet(
    os.path.join(RAW_DATA_PATH, "yellow_tripdata_2025-01.parquet")
).head(1000)

green_sample = pd.read_parquet(
    os.path.join(RAW_DATA_PATH, "green_tripdata_2025-01.parquet")
).head(1000)

# ── Shape ──────────────────────────────────────────────────────
print("YELLOW - rows and columns:", yellow_sample.shape)
print("GREEN  - rows and columns:", green_sample.shape)

# ── Column names and data types ────────────────────────────────
print("\n── YELLOW COLUMNS ──")
print(yellow_sample.dtypes)

print("\n── GREEN COLUMNS ──")
print(green_sample.dtypes)

# ── Sample rows ────────────────────────────────────────────────
print("\n── YELLOW SAMPLE ──")
display(yellow_sample.head(3))

print("\n── GREEN SAMPLE ──")
display(green_sample.head(3))

# ── Missing values ─────────────────────────────────────────────
print("\n── YELLOW NULL COUNTS ──")
print(yellow_sample.isnull().sum())

print("\n── GREEN NULL COUNTS ──")
print(green_sample.isnull().sum())

YELLOW - rows and columns: (1000, 20)
GREEN  - rows and columns: (1000, 21)

── YELLOW COLUMNS ──
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

── GREEN COLUMNS ──
VendorID                          int32
lpep_pickup_datetime     da

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.6,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.5,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.6,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0



── GREEN SAMPLE ──


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,...,0.5,6.8,0.0,NaN,1.0,34.00,1.0,1.0,0.0,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,...,0.5,0.0,0.0,NaN,1.0,11.10,2.0,1.0,0.0,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,...,0.0,0.0,0.0,NaN,1.0,26.55,2.0,2.0,0.0,0.0



── YELLOW NULL COUNTS ──
VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
dtype: int64

── GREEN NULL COUNTS ──
VendorID                    0
lpep_pickup_datetime        0
lpep_dropoff_datetime       0
store_and_fwd_flag          0
RatecodeID                  0
PULocationID                0
DOLocationID                0
passenger_count             0
trip_distance               0
fare_amount                 0
extra                       0
mta_tax                     0
tip_amount                  0
tolls_

## Step 3 - Clean and Standardize the Data
We clean each file and standardize the schema so Yellow and Green
trips can be combined into a single unified table.

Key transformations:
- Rename datetime columns to a common name
- Standardize column names to lowercase
- Add taxi_type column to identify the source
- Calculate trip_duration_min from pickup and dropoff times
- Drop columns not in our schema (ehail_fee)
- Convert float columns to integers where appropriate
- Remove invalid rows (negative fares, zero distance, bad timestamps)

In [6]:
def clean_taxi_data(df, taxi_type):
    """
    Clean and standardize a raw TLC taxi dataframe.
    Works for both Yellow and Green taxi data.
    """

    # Rename datetime columns to a common name
    df = df.rename(columns={
        "tpep_pickup_datetime"  : "pickup_datetime",
        "tpep_dropoff_datetime" : "dropoff_datetime",
        "lpep_pickup_datetime"  : "pickup_datetime",
        "lpep_dropoff_datetime" : "dropoff_datetime"
    })

    # Standardize all column names to lowercase
    df.columns = df.columns.str.lower()

    # Add taxi_type column and calculate trip duration in minutes
    df["taxi_type"]         = taxi_type
    df["trip_duration_min"] = ((df["dropoff_datetime"] - df["pickup_datetime"])
                                .dt.total_seconds() / 60).round(2)

    # Add trip_type column for Yellow rows (Green already has it)
    if "trip_type" not in df.columns:
        df["trip_type"] = None

    # Drop columns not in our schema
    df = df.drop(columns=[c for c in ["ehail_fee", "dropoff_datetime"] if c in df.columns])

    # Remove invalid rows
    original_count = len(df)
    df = df[(df["trip_distance"]     >  0) &
            (df["fare_amount"]       >= 0) &
            (df["total_amount"]      >= 0) &
            (df["trip_duration_min"] >  0) &
            (df["trip_duration_min"] <  300)]
    print(f"  Removed {original_count - len(df)} invalid rows")

    return df


# ── Test on January 2025 ────────────────────────────────────────
yellow_clean = clean_taxi_data(
    pd.read_parquet(os.path.join(RAW_DATA_PATH, "yellow_tripdata_2025-01.parquet")),
    "yellow"
)

green_clean = clean_taxi_data(
    pd.read_parquet(os.path.join(RAW_DATA_PATH, "green_tripdata_2025-01.parquet")),
    "green"
)

print(f"\nYellow clean shape : {yellow_clean.shape}")
print(f"Green  clean shape : {green_clean.shape}")
display(yellow_clean.head(3))
display(green_clean.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\hedye\\Desktop\\Drive D\\Courses I took\\GitHub\\data-analytics-portfolio\\SQL\\Raw Dataset\\yellow_tripdata_2025-01.parquet'